In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageFilter, ImageFile
import cv2
from skimage import filters, feature, color
from skimage.measure import shannon_entropy
import warnings

### Load Metadata

In [3]:
df_images = pd.read_csv("/Users/jana/Documents/Github/Data_Mining_Project/dataset/processed/image_metadata.csv")
print(df_images.shape)
df_images.head()

(60000, 5)


,path,filename,split,label,ext
0,/Users/jana/Documents/Github/Data_Mining_Proje...,0071.jpg,test,real,.jpg
1,/Users/jana/Documents/Github/Data_Mining_Proje...,4217.jpg,test,real,.jpg
2,/Users/jana/Documents/Github/Data_Mining_Proje...,3578.jpg,test,real,.jpg
3,/Users/jana/Documents/Github/Data_Mining_Proje...,2666.jpg,test,real,.jpg
4,/Users/jana/Documents/Github/Data_Mining_Proje...,5109.jpg,test,real,.jpg


In [8]:
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.simplefilter("ignore", Image.DecompressionBombWarning)


def extract_image_features(img_path, resize_to=(256, 256)):
    try:
        img = Image.open(img_path).convert("RGB")
        original_width, original_height = img.size
        
        # resize for faster and standardized feature extraction
        img = img.resize(resize_to)
        arr = np.array(img)
        
        height, width, channels = arr.shape
        aspect_ratio = original_width / original_height if original_height != 0 else np.nan
        
        # RGB stats
        mean_r = arr[:, :, 0].mean()
        mean_g = arr[:, :, 1].mean()
        mean_b = arr[:, :, 2].mean()
        
        std_r = arr[:, :, 0].std()
        std_g = arr[:, :, 1].std()
        std_b = arr[:, :, 2].std()
        
        # grayscale
        gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
        
        brightness = gray.mean()
        contrast = gray.std()
        entropy = shannon_entropy(gray)
        
        # sharpness
        sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()
        
        # edge density
        edges = cv2.Canny(gray, 100, 200)
        edge_density = edges.mean() / 255.0
        
        # colorfulness
        rg = arr[:, :, 0].astype(float) - arr[:, :, 1].astype(float)
        yb = 0.5 * (arr[:, :, 0].astype(float) + arr[:, :, 1].astype(float)) - arr[:, :, 2].astype(float)
        colorfulness = np.sqrt(rg.std()**2 + yb.std()**2) + 0.3 * np.sqrt(rg.mean()**2 + yb.mean()**2)
        
        return {
            "orig_width": original_width,
            "orig_height": original_height,
            "width": width,
            "height": height,
            "aspect_ratio": aspect_ratio,
            "mean_r": mean_r,
            "mean_g": mean_g,
            "mean_b": mean_b,
            "std_r": std_r,
            "std_g": std_g,
            "std_b": std_b,
            "brightness": brightness,
            "contrast": contrast,
            "entropy": entropy,
            "sharpness": sharpness,
            "edge_density": edge_density,
            "colorfulness": colorfulness,
            "error": None
        }
    
    except Exception as e:
        return {"error": str(e)}

In [ ]:
sample_test = df_images.sample(5, random_state=42).copy()

sample_features = []
for _, row in sample_test.iterrows():
    feats = extract_image_features(row["path"])
    sample_features.append({
        "path": row["path"],
        "label": row["label"],
        **feats
    })

pd.DataFrame(sample_features)

### Extract features for all images

In [12]:
rows = []

for i, row in df_images.iterrows():
    feats = extract_image_features(row["path"], resize_to=(256, 256))
    
    rows.append({
        "path": row["path"],
        "filename": row["filename"],
        "split": row["split"],
        "label": row["label"],
        "ext": row["ext"],
        **feats
    })
    
    if (i + 1) % 500 == 0:
        print(f"Processed {i+1:,} / {len(df_images):,}")

Processed 500 / 60,000
Processed 1,000 / 60,000
Processed 1,500 / 60,000
Processed 2,000 / 60,000
Processed 2,500 / 60,000
Processed 3,000 / 60,000
Processed 3,500 / 60,000
Processed 4,000 / 60,000
Processed 4,500 / 60,000
Processed 5,000 / 60,000
Processed 5,500 / 60,000
Processed 6,000 / 60,000
Processed 6,500 / 60,000
Processed 7,000 / 60,000
Processed 7,500 / 60,000
Processed 8,000 / 60,000
Processed 8,500 / 60,000
Processed 9,000 / 60,000
Processed 9,500 / 60,000
Processed 10,000 / 60,000
Processed 10,500 / 60,000
Processed 11,000 / 60,000
Processed 11,500 / 60,000
Processed 12,000 / 60,000
Processed 12,500 / 60,000
Processed 13,000 / 60,000
Processed 13,500 / 60,000
Processed 14,000 / 60,000
Processed 14,500 / 60,000
Processed 15,000 / 60,000
Processed 15,500 / 60,000
Processed 16,000 / 60,000
Processed 16,500 / 60,000
Processed 17,000 / 60,000
Processed 17,500 / 60,000
Processed 18,000 / 60,000
Processed 18,500 / 60,000
Processed 19,000 / 60,000
Processed 19,500 / 60,000
Process

In [13]:
df_features = pd.DataFrame(rows)
print(df_features.shape)
df_features.head()

(60000, 23)


,path,filename,split,label,ext,orig_width,orig_height,width,height,aspect_ratio,...,std_r,std_g,std_b,brightness,contrast,entropy,sharpness,edge_density,colorfulness,error
0,/Users/jana/Documents/Github/Data_Mining_Proje...,0071.jpg,test,real,.jpg,800,455,256,256,1.758242,...,78.846701,80.070388,71.296612,121.370941,77.573049,7.922745,2648.262433,0.198181,55.298258,None
1,/Users/jana/Documents/Github/Data_Mining_Proje...,4217.jpg,test,real,.jpg,1080,1620,256,256,0.666667,...,36.932017,41.233024,53.079293,176.241531,39.498801,6.731529,231.760054,0.028976,42.816157,None
2,/Users/jana/Documents/Github/Data_Mining_Proje...,3578.jpg,test,real,.jpg,1080,720,256,256,1.500000,...,48.873092,41.627177,43.153407,66.234970,41.699535,7.251361,1809.504625,0.206909,41.967958,None
3,/Users/jana/Documents/Github/Data_Mining_Proje...,2666.jpg,test,real,.jpg,1080,1620,256,256,0.666667,...,20.886327,22.823957,15.364015,32.806351,20.987434,5.714562,2008.046290,0.141129,17.450586,None
4,/Users/jana/Documents/Github/Data_Mining_Proje...,5109.jpg,test,real,.jpg,1449,1069,256,256,1.355472,...,20.333942,16.172023,6.890060,52.705078,16.113609,5.632007,101.842324,0.021729,24.015448,None


In [14]:
print(df_features.isna().sum().sort_values(ascending=False).head(20))

if "error" in df_features.columns:
    print(df_features["error"].value_counts(dropna=False).head(10))

error           60000
mean_b              0
colorfulness        0
edge_density        0
sharpness           0
entropy             0
contrast            0
brightness          0
std_b               0
std_g               0
std_r               0
path                0
filename            0
mean_r              0
aspect_ratio        0
height              0
width               0
orig_height         0
orig_width          0
ext                 0
dtype: int64
error
None    60000
Name: count, dtype: int64


In [15]:
if "error" in df_features.columns:
    df_features = df_features[df_features["error"].isna()].copy()

print(df_features.shape)

(60000, 23)


In [16]:
from pathlib import Path

output_path = Path("/Users/jana/Documents/Github/Data_Mining_Project/dataset/processed/image_features.csv")
df_features.to_csv(output_path, index=False)
print("Saved:", output_path)

Saved: /Users/jana/Documents/Github/Data_Mining_Project/dataset/processed/image_features.csv
